# RE_304 · Quina lletra traurà el meu pis? · resolt

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Treballes en una empresa petita de rehabilitació energètica. Volen posar a la web una calculadora:
el propietari hi entra les dades del seu habitatge **que pot saber sense contractar cap tècnic**
(població, zona climàtica, any de construcció, metres, tipus d'edifici...) i la calculadora li diu
quina **lletra de consum** (de la A a la G) és probable que tregui al certificat energètic. És el
repte més complet del bloc: aquí ningú no et dirà què està brut.

**La pregunta:** quin F1 *macro* es pot prometre, honestament, amb només aquestes dades? I on s'equivoca?

L'enunciat complet, amb el diccionari de columnes, és a [RE_304](../RE_304_certificats_energetics.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Còpia congelada al repositori del curs (les dades originals són vives i canvien).
# Si tens el repositori clonat, llegeix la còpia local; si no (Colab), la baixa de GitHub.
DADES = "https://raw.githubusercontent.com/ZenidX/Machine-Learning/main/Machine%20Learning/04_reptes/dades/"

def llegeix(fitxer, **opcions):
    for ruta in ("dades/" + fitxer, "../dades/" + fitxer):
        if os.path.exists(ruta):
            return pd.read_csv(ruta, **opcions)
    return pd.read_csv(DADES + fitxer, **opcions)

df = llegeix("certificats_energetics_mostra.csv")
print(df.shape)
df.head()

In [ ]:
from sklearn.compose import make_column_transformer, make_column_selector
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

OBJ = "qualificaci_de_consum_d"
y = df[OBJ]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
num = make_column_selector(dtype_include="number")
txt = make_column_selector(dtype_exclude="number")
def prepara():
    return make_column_transformer(
        (SimpleImputer(strategy="median"), num),
        (make_pipeline(SimpleImputer(strategy="most_frequent"),
                       OneHotEncoder(handle_unknown="ignore", min_frequency=30)), txt))
def bosc():
    return RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)

## 1.1 · La foto

Forma, tipus, nuls i duplicats. Com es reparteixen les lletres? Quin F1 *macro* tindria un model
que digués sempre la lletra més freqüent? I un que les digués a l'atzar respectant les proporcions?

In [ ]:
print("duplicats:", df.duplicated().sum())
print((df.isna().mean() * 100).round(1)[lambda s: s > 0].to_dict())
lletres = y.value_counts().sort_index()
assert lletres.sum() == 30000
print(lletres.to_dict())
X0 = df.drop(columns=[OBJ])
ref_f1 = {}
for estr in ["most_frequent", "stratified"]:
    d = DummyClassifier(strategy=estr, random_state=0).fit(X0, y)
    p = d.predict(X0)
    ref_f1[estr] = f1_score(y, p, average="macro")
    print(estr, "accuracy", round((p == y).mean(), 3), "F1 macro", round(ref_f1[estr], 3))
assert ref_f1["most_frequent"] < (y == "E").mean()

La lletra **E** és el 44 % de la mostra. El model que diu sempre E té un 44 % d'*accuracy* però un
F1 *macro* de 0,09 (les altres sis lletres hi compten amb zero); a l'atzar amb proporcions,
uns 0,14. Hi ha 136 files duplicades (no hi ha identificador: poden ser pisos iguals d'un mateix
bloc).

## 1.2 · Les columnes de text

Per a cada columna de text, quantes categories té? Mira de prop les que en tenen més d'una
dotzena: **quantes categories reals** hi ha, si les llegeixes com ho faria una persona? Escriu una
funció que les ajunti.

In [ ]:
text = df.select_dtypes(exclude="number").columns.drop([OBJ, "data_entrada"])
df[text].nunique().sort_values(ascending=False)

In [ ]:
print(df["motiu_de_la_certificacio"].value_counts().to_string())

In [ ]:
def motiu_net(s):
    s = s.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    regles = [  # (si conté..., categoria)
        ("compravend|compravent", "compravenda"),
        ("lloguer|alquiler", "lloguer"),
        ("volunt", "voluntària"),
        ("ajut|ayuda", "ajuts"),
        ("nova construcci|nueva construcci", "nova construcció"),
        ("reform|ampliaci", "reforma o ampliació"),
        ("administraci", "edifici públic"),
        ("renovaci", "renovació"),
        (r"\(iee\)|\(ite\)|avaluaci|evaluaci", "IEE / ITE"),
        ("500 m", "+500 m² ús públic"),
    ]
    out = pd.Series(pd.NA, index=s.index, dtype="object")
    for patro, cat in regles:
        out = out.where(out.notna() | ~s.str.contains(patro, regex=True, na=False), cat)
    return out.fillna("altres").where(s.notna())

def us_net(s):
    s = s.str.lower()
    return np.select([s.str.contains("unifamiliar"), s.str.contains("individual"),
                      s.str.contains("bloc|bloq"), s.str.contains("terciari")],
                     ["unifamiliar", "pis en bloc", "bloc sencer", "terciari"], "altres")

def normativa_net(s):
    return s.replace({"Antes de 1979": "Abans de 1979", "0": np.nan})

m = motiu_net(df["motiu_de_la_certificacio"])
u = pd.Series(us_net(df["us_edifici"]), index=df.index)
n = normativa_net(df["normativa_construcci"])
for abans, despres in [(df["motiu_de_la_certificacio"], m), (df["us_edifici"], u), (df["normativa_construcci"], n)]:
    assert despres.nunique() < abans.nunique()
    assert despres.value_counts(dropna=False).sum() == 30000
print(m.value_counts(dropna=False).to_dict())
print(u.value_counts().to_dict())
print(df["motiu_de_la_certificacio"].nunique(), "->", m.nunique(), "|", df["us_edifici"].nunique(), "->", u.nunique())

`motiu_de_la_certificacio` té desenes de valors per a una desena de conceptes: el mateix motiu en
**català i castellà** (`Compravenda` / `Compraventa`, `Lloguer` / `Alquiler`...), amb apòstrofs
tipogràfics i rectes, dobles espais, `m2` i `m²`, i faltes (`Sol·licitut`). `us_edifici` té 10
valors per a 4 conceptes (també bilingüe: `Habitatge unifamiliar` / `Vivienda unifamiliar`...), i `normativa_construcci` té `Abans` / `Antes` i un `0`.
La zona climàtica té un `alpha3` i l'eina de certificació un `0`: soroll de poques files.

## 1.3 · Els números plausibles

Mira el mínim i el màxim de cada columna numèrica que podria conèixer el propietari. Quins valors
són impossibles o molt sospitosos? Decideix uns límits i converteix el que en quedi fora en `NaN`
(no esborris files).

In [ ]:
print(df[["any_construccio", "metres_cadastre", "valor_aillaments", "valor_finestres"]].describe().round(1).to_string())
print("anys < 1700:", (df["any_construccio"] < 1700).sum(), "| metres > 5000:", (df["metres_cadastre"] > 5000).sum())

In [ ]:
LIMITS = {"any_construccio": (1700, 2026), "metres_cadastre": (15, 5000)}
def plausibles(X):
    X = X.copy()
    for col, (lo, hi) in LIMITS.items():
        X[col] = X[col].where(X[col].between(lo, hi))
    return X
p = plausibles(df)
assert len(p) == len(df)
for col, (lo, hi) in LIMITS.items():
    assert p[col].dropna().between(lo, hi).all()
print("valors convertits en NaN:", {c: int(p[c].isna().sum() - df[c].isna().sum()) for c in LIMITS})

Hi ha anys de construcció impossibles (el mínim de la mostra és **970**; a la base sencera n'hi ha
d'**1**) i superfícies de desenes de milers de m² (el màxim, 35.849 m²: no és un habitatge). Es converteixen en `NaN` i l'imputador ja farà la seva feina. Els límits
són una decisió: s'han de dir. `valor_finestres` també té valors absurds, però és un resultat del
certificat (vegeu 2.2).

## 2.1 · El primer model, amb tot

Entrena un bosc aleatori amb totes les columnes (menys l'objectiu i `data_entrada`) dins d'un
`Pipeline`. Quin F1 *macro* treu amb validació creuada de 5 trossos estratificats? Compara'l amb la
referència.

In [ ]:
X_tot = df.drop(columns=[OBJ, "data_entrada"])
f1_tot = cross_val_score(make_pipeline(prepara(), bosc()), X_tot, y, cv=cv, scoring="f1_macro", n_jobs=1)
assert np.isnan(f1_tot).sum() == 0
print(f1_tot.round(3), round(f1_tot.mean(), 3))

## 2.2 · Què sabrà el propietari, i què no

Torna a l'encàrrec: el propietari **no té** el certificat. Fes la llista de columnes que **surten
del càlcul del certificat** i que, per tant, la calculadora no tindrà. Comprova cada sospita mirant
com es reparteix aquella columna dins de cada lletra. Entrena el mateix model sense elles.

In [ ]:
sospitoses = ["energia_prim_ria_no_renovable", "consum_d_energia_final", "emissions_de_co2",
              "cost_anual_aproximat_d_energia", "valor_aillaments", "valor_finestres", "ventilacio_us_residencial"]
df.groupby(OBJ)[sospitoses].median().round(1)

In [ ]:
pd.crosstab(df[OBJ], df["qualificacio_d_emissions"])

In [ ]:
DEL_CERTIFICAT = {
    "energia_prim_ria_no_renovable": "és el número amb què es calcula la lletra (la defineix)",
    "consum_d_energia_final": "consum calculat pel certificador",
    "emissions_de_co2": "es calcula a partir del consum",
    "qualificacio_d_emissions": "l'altra lletra del mateix certificat",
    "cost_anual_aproximat_d_energia": "es calcula a partir del consum",
    "valor_aillaments": "valoració de l'envolupant que fa el tècnic",
    "valor_finestres": "valoració de les obertures que fa el tècnic",
    "ventilacio_us_residencial": "paràmetre del càlcul (renovacions/hora)",
    "eina_de_certificacio": "programa que fa servir el tècnic",
    "tipus_tramit": "el tràmit del certificat",
}
X_hon = df.drop(columns=[OBJ, "data_entrada"] + list(DEL_CERTIFICAT))
assert set(DEL_CERTIFICAT) & set(X_hon.columns) == set()
f1_hon_brut = cross_val_score(make_pipeline(prepara(), bosc()), X_hon, y, cv=cv, scoring="f1_macro")
print(list(X_hon.columns))
print(round(f1_hon_brut.mean(), 3))

Les medianes d'energia primària, consum, emissions i cost **creixen sense excepció** de la A a la
G, i la lletra d'emissions coincideix amb la de consum a la diagonal: són resultats del mateix
càlcul. El propietari no els té sense un tècnic. Els valors d'aïllament i finestres també són
avaluació del tècnic. Es treuen totes.

## 2.3 · El Pipeline que neteja sol

Posa la neteja de la 1.2 i la 1.3 **dins** del `Pipeline` (amb `FunctionTransformer`), de manera
que el `Pipeline` accepti la taula **tal com es carrega**. Avalua'l amb la mateixa validació creuada.

In [ ]:
def neteja(X):
    X = plausibles(X)
    X["motiu_de_la_certificacio"] = motiu_net(X["motiu_de_la_certificacio"])
    X["us_edifici"] = us_net(X["us_edifici"])
    X["normativa_construcci"] = normativa_net(X["normativa_construcci"])
    X["zona_climatica"] = X["zona_climatica"].where(X["zona_climatica"].str.fullmatch(r"[A-E][1-4]", na=False))
    X["codi_postal"] = X["codi_postal"].astype("Int64").astype(str).str.zfill(5).str[:3].where(X["codi_postal"].notna())
    return X

X_cru = df.drop(columns=[OBJ, "data_entrada"] + list(DEL_CERTIFICAT))
pipe = make_pipeline(FunctionTransformer(neteja), prepara(), bosc())
pipe.fit(X_cru, y)          # accepta la taula tal com es carrega
f1_hon = cross_val_score(pipe, X_cru, y, cv=cv, scoring="f1_macro")
print(f1_hon.round(3), round(f1_hon.mean(), 3), "(abans de netejar:", round(f1_hon_brut.mean(), 3), ")")

La neteja dins del `Pipeline` es torna a aplicar a cada tros de validació i, el dia de demà, a
cada pis nou que entri a la calculadora: no cal recordar-la. El codi postal es redueix als tres
primers dígits (zona), perquè el codi sencer té massa categories i s'acosta massa a una adreça.

## 2.4 · On s'equivoca

Fes la matriu de confusió del model final (amb una partició d'entrenament i test). Quan s'equivoca,
s'equivoca per una lletra o per moltes? Quines lletres confon més, i per què pot ser?

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_cru, y, test_size=0.25, stratify=y, random_state=0)
pipe.fit(X_tr, y_tr)
y_pred = pipe.predict(X_te)
disp = ConfusionMatrixDisplay.from_predictions(y_te, y_pred, labels=list("ABCDEFG"))
assert disp.confusion_matrix.shape == (7, 7) and disp.confusion_matrix.sum() == len(X_te)
pos = lambda s: s.map("ABCDEFG".index)
dist = (pos(pd.Series(y_pred, index=y_te.index)) - pos(y_te)).abs()
print("encert:", round((dist == 0).mean(), 3), "| a una lletra o menys:", round((dist <= 1).mean(), 3),
      "| distància mitjana:", round(dist.mean(), 2))

Encerta la lletra exacta la meitat de les vegades, però **la lletra o la del costat** en tres de cada
quatre: més de la meitat dels errors són a una sola lletra. El model tendeix a dir E (la
majoritària). Les lletres poc freqüents (B, C) són les que pitjor encerta: és el que fa baixar el
F1 *macro*.

## 2.5 · La resposta

Respon l'encàrrec: quin F1 *macro* es pot prometre, quant més alt sortia amb el primer model, i
per què el primer no valia. Afegeix una frase sobre si la calculadora és útil tal com està.

In [ ]:
res = pd.Series({"referència (sempre E)": ref_f1["most_frequent"],
                 "primer model (amb el certificat)": f1_tot.mean(),
                 "honest, sense netejar": f1_hon_brut.mean(),
                 "honest, amb neteja al Pipeline": f1_hon.mean()}).round(3)
assert f1_tot.mean() > 0.8 and f1_hon.mean() < 0.5
res

**Resposta:** amb només les dades que té el propietari, el F1 *macro* és d'uns **0,4** (5 trossos
estratificats). El primer model donava més de 0,8 perquè llegia el resultat del mateix
certificat: era una fuita, no un model. La calculadora pot orientar (encerta la lletra o la del
costat la major part de les vegades), però no substitueix el certificat. La neteja de categories
no mou el número (0,40 abans i després, dins del soroll entre trossos: un bosc ja tolera
categories repetides amb noms diferents, i reduir el codi postal a la zona hi resta una mica de
detall); el que el mou de debò és treure les columnes del certificat. La neteja dins del
`Pipeline` val per una altra raó: la calculadora rebrà dades brutes i les ha d'entendre.